# Solutions generation

Generating solutions of ill-defined problems using different LLM-based baselines.

In [14]:
PROBLEMS_FILE = './data/problems.json'
SOLUTIONS_FILE = './data/solutions.json'
MAX_WORKERS = 5
MODEL = 'deepseek/deepseek-v4-flash'
TEMPERATURE = 0.5

In [15]:
from src import read_json, write_json, generate_id

problems = read_json(PROBLEMS_FILE)
solutions = read_json(SOLUTIONS_FILE) or {}

In [16]:
from langchain_openai import ChatOpenAI
from fp2mp_baselines.config import config

llm = ChatOpenAI(
    model=MODEL,
    base_url=config.base_url,
    api_key=config.api_key,
    temperature=TEMPERATURE,
)

In [17]:
from fp2mp_core.tools.code_exec import execute_python_tool, list_available_libraries_tool, check_available_data_tool 
from fp2mp_core.tools.web_search import fetch_url_tool, web_search_tool

tools = [
    execute_python_tool, list_available_libraries_tool, check_available_data_tool,
    fetch_url_tool, web_search_tool
]

from fp2mp_baselines.single_agent import build_single_agent_graph
from fp2mp_baselines.cot import build_cot_graph
from fp2mp_baselines.react import build_react_graph
from fp2mp_baselines.generator_critic import build_generator_critic_graph
from fp2mp_baselines.debate import build_debate_graph
from fp2mp_baselines.major_vote import build_major_vote_graph

baselines = {
    'single-agent' : build_single_agent_graph(llm),
    'chain-of-thoughts': build_cot_graph(llm),
    # 'react': build_react_graph(llm, tools=tools), 
    'generator-critic': build_generator_critic_graph(llm),
    'debate': build_debate_graph(llm, num_agents=3, debate_rounds=3),
    'major-vote': build_major_vote_graph(llm, num_agents=3)
}

Creating tasks for problems-baselines pairs not presented in `solutions.json`

In [18]:
tasks = []

for problem_id in problems.keys():
    for baseline_name in baselines.keys():
        solution_id = generate_id(problem_id, baseline_name, MODEL)
        if solution_id not in solutions:
            tasks.append({
                'solution_id': solution_id,
                'problem_id': problem_id,
                'baseline_name': baseline_name
            })

len(tasks)

238

In [19]:
from tqdm.contrib.concurrent import thread_map

def solve_task(task : dict[str,str]):
    solution_id = task['solution_id']
    problem_id = task['problem_id']
    baseline_name = task['baseline_name']

    problem_content = problems[problem_id]['content']
    baseline_graph = baselines[baseline_name]

    graph_state = baseline_graph.invoke({'input': problem_content})
    return {
        'solution_id': solution_id,
        'solution_data': {
            'problem_id': problem_id,
            'baseline': baseline_name,
            'model': MODEL,
            'content': graph_state['output'],
            'log': [m.model_dump() for m in graph_state['log']]
        }
    }    

results = thread_map(solve_task, tasks, max_workers=MAX_WORKERS)

  0%|          | 0/238 [00:00<?, ?it/s]

In [20]:
for result in results:
    solution_id = result['solution_id']
    solution_data = result['solution_data']
    solutions[solution_id] = solution_data

write_json(solutions, SOLUTIONS_FILE)